# DynamicPD-CLO Raw Data Audit

This notebook checks raw extraction quality before feature engineering begins. It audits:

- extraction completeness
- schema and key integrity
- date coverage
- duplicate rows
- identifier consistency
- mergeability
- potential timing/leakage issues

This notebook uses only local Parquet and JSON files. It does not query WRDS or any external API, does not modify raw data, and does not perform feature engineering.

In [23]:
# Imports and paths
from pathlib import Path
import json
import pandas as pd
import numpy as np

try:
    import pyarrow.parquet as pq
except Exception as exc:
    pq = None
    print(f"PyArrow metadata unavailable: {exc}")

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent.resolve()

RAW = ROOT / "data" / "raw"
WRDS = RAW / "wrds"
MACRO = RAW / "macro"
PATHS = {
    "wrds_raw": WRDS,
    "fisd_rating_hist": WRDS / "fisd_rating_hist",
    "fisd_issue_master": WRDS / "fisd_rated_issue_master.parquet",
    "fisd_issuer_master": WRDS / "fisd_rated_issuer_master.parquet",
    "fisd_issue_issuer_map": WRDS / "fisd_issue_issuer_map.parquet",
    "fisd_extra": WRDS / "fisd_extra",
    "compustat_quarterly": WRDS / "compustat_quarterly",
    "compustat_annual": WRDS / "compustat_annual",
    "crsp_monthly": WRDS / "crsp" / "monthly",
    "crsp_daily": WRDS / "crsp" / "daily",
    "crsp_market_monthly": WRDS / "crsp" / "market_monthly",
    "crsp_market_daily": WRDS / "crsp" / "market_daily",
    "crsp_names": WRDS / "crsp" / "crsp_names.parquet",
    "crsp_delistings": WRDS / "crsp" / "crsp_delistings.parquet",
    "crsp_ccm": WRDS / "crsp" / "crsp_compustat_link.parquet",
    "bondcrsp_link": WRDS / "bondcrsp_link.parquet",
    "bondret_std": WRDS / "bondret_std",
    "bondret": WRDS / "bondret",
    "ibes": WRDS / "ibes",
    "macro_raw": MACRO,
    "macro_manifest": MACRO / "extraction_manifest.json",
    "macro_treasury_rates": MACRO / "treasury_rates.parquet",
    "macro_gdp_growth": MACRO / "gdp_growth.parquet",
    "macro_fed_funds": MACRO / "fed_funds.parquet",
    "macro_unemployment": MACRO / "unemployment.parquet",
    "macro_cpi": MACRO / "cpi.parquet",
    "macro_credit_spreads": MACRO / "credit_spreads.parquet",
    "macro_vix": MACRO / "vix.parquet",
}

MACRO_DATASETS = {
    "treasury_rates": {
        "path": PATHS["macro_treasury_rates"],
        "columns": ["treasury_3m", "treasury_10y", "term_spread"],
        "series_ids": ["DGS3MO", "DGS10"],
    },
    "gdp_growth": {
        "path": PATHS["macro_gdp_growth"],
        "columns": ["gdp_growth_qoq"],
        "series_ids": ["A191RL1Q225SBEA"],
    },
    "fed_funds": {
        "path": PATHS["macro_fed_funds"],
        "columns": ["fed_funds_rate"],
        "series_ids": ["FEDFUNDS"],
    },
    "unemployment": {
        "path": PATHS["macro_unemployment"],
        "columns": ["unemployment_rate", "unemployment_change"],
        "series_ids": ["UNRATE"],
    },
    "cpi": {
        "path": PATHS["macro_cpi"],
        "columns": ["cpi", "cpi_yoy"],
        "series_ids": ["CPIAUCSL"],
    },
    "credit_spreads": {
        "path": PATHS["macro_credit_spreads"],
        "columns": ["aaa10y", "baa10y"],
        "series_ids": ["AAA10Y", "BAA10Y"],
    },
    "vix": {
        "path": PATHS["macro_vix"],
        "columns": ["vix"],
        "series_ids": ["VIXCLS"],
    },
}

print(f"Project root: {ROOT}")


Project root: /Users/harshkulkarni/Documents/Projects/DynamicPD-CLO


In [24]:
# Helper functions
AUDIT_FLAGS = []

def flag(dataset, severity, issue, detail=""):
    AUDIT_FLAGS.append({"dataset": dataset, "severity": severity, "issue": issue, "detail": detail})


def display_compact(df, n=20, title=None):
    if title:
        print(title)
    if df is None:
        print("No data")
        return
    if not isinstance(df, pd.DataFrame):
        display(df)
        return
    display(df.head(n) if len(df) > n else df)
    if len(df) > n:
        print(f"Showing {n} of {len(df):,} rows")


def parquet_files(path, pattern="*.parquet"):
    path = Path(path)
    if path.is_file() and path.suffix == ".parquet":
        return [path]
    if not path.exists():
        return []
    return sorted(path.glob(pattern))


def year_from_name(path):
    import re
    matches = re.findall(r"(19\d{2}|20\d{2})", Path(path).stem)
    return int(matches[-1]) if matches else None


def file_size_mb(path):
    path = Path(path)
    if path.is_file():
        return path.stat().st_size / 1_000_000
    if path.is_dir():
        return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / 1_000_000
    return 0.0


def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as exc:
        flag(str(path), "FAIL", "JSON read error", str(exc))
        return None


def parquet_metadata(path):
    path = Path(path)
    out = {"path": str(path), "exists": path.exists(), "rows": np.nan, "columns": np.nan, "error": None}
    if not path.exists():
        return out
    try:
        if pq is not None:
            meta = pq.ParquetFile(path).metadata
            out.update({"rows": int(meta.num_rows), "columns": int(meta.num_columns)})
        else:
            df = pd.read_parquet(path)
            out.update({"rows": len(df), "columns": len(df.columns)})
    except Exception as exc:
        out["error"] = str(exc)
        flag(path.name, "FAIL", "Parquet metadata/read error", str(exc))
    return out


def safe_read(path, columns=None):
    path = Path(path)
    try:
        return pd.read_parquet(path, columns=columns)
    except Exception as exc:
        flag(path.name, "FAIL", "Parquet read error", str(exc))
        return pd.DataFrame()


def safe_concat(paths, columns=None, max_files=None):
    frames = []
    for p in list(paths)[:max_files]:
        df = safe_read(p, columns=columns)
        if not df.empty:
            df["__source_file"] = Path(p).name
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def min_max_date(df, col):
    if df.empty or col not in df.columns:
        return (pd.NaT, pd.NaT)
    s = pd.to_datetime(df[col], errors="coerce").dropna()
    return (s.min(), s.max()) if not s.empty else (pd.NaT, pd.NaT)


def duplicate_count(df, cols=None):
    if df.empty:
        return 0
    if cols is None:
        return int(df.duplicated().sum())
    missing = [c for c in cols if c not in df.columns]
    if missing:
        return np.nan
    return int(df.duplicated(cols).sum())


def missingness(df, cols):
    rows = []
    for c in cols:
        rows.append({"column": c, "missing_rate": np.nan if c not in df.columns or len(df)==0 else float(df[c].isna().mean()), "exists": c in df.columns})
    return pd.DataFrame(rows)


def value_counts_table(df, col, n=20):
    if df.empty or col not in df.columns:
        return pd.DataFrame(columns=[col, "count"])
    return df[col].value_counts(dropna=False).head(n).rename_axis(col).reset_index(name="count")


def partition_summary(files, date_col=None, id_cols=()):
    rows = []
    for p in files:
        meta = parquet_metadata(p)
        row = {"file": p.name, "year": year_from_name(p), "rows": meta["rows"], "columns": meta["columns"], "error": meta["error"]}
        cols = [c for c in set([date_col, *id_cols]) if c]
        if cols and meta["error"] is None:
            df = safe_read(p, columns=list(cols))
            if date_col in df.columns:
                mn, mx = min_max_date(df, date_col)
                row.update({"min_date": mn, "max_date": mx})
            for c in id_cols:
                if c in df.columns:
                    row[f"distinct_{c}"] = df[c].nunique(dropna=True)
            if id_cols and all(c in df.columns for c in id_cols):
                row["duplicate_key_rows"] = duplicate_count(df, list(id_cols))
        rows.append(row)
    return pd.DataFrame(rows)


def expected_missing_years(files, start, end):
    actual = sorted({year_from_name(p) for p in files if year_from_name(p) is not None})
    expected = list(range(start, end + 1))
    missing = sorted(set(expected) - set(actual))
    return expected, actual, missing


def path_inventory_row(name, path, expected_years=None):
    path = Path(path)
    files = parquet_files(path)
    actual_years = sorted({year_from_name(p) for p in files if year_from_name(p) is not None})
    missing = sorted(set(expected_years or []) - set(actual_years)) if expected_years else []
    manifest = path / "extraction_manifest.json" if path.is_dir() else path.parent / "extraction_manifest.json"
    audit = path / "column_audit.json" if path.is_dir() else path.parent / "column_audit.json"
    return {
        "dataset": name,
        "path": str(path.relative_to(ROOT)) if path.exists() else str(path),
        "exists": path.exists(),
        "parquet_files": len(files),
        "total_file_size_mb": round(file_size_mb(path), 2),
        "manifest_exists": manifest.exists(),
        "column_audit_exists": audit.exists(),
        "expected_years": f"{min(expected_years)}-{max(expected_years)}" if expected_years else "",
        "actual_years": f"{min(actual_years)}-{max(actual_years)}" if actual_years else "",
        "missing_years": missing[:20],
    }

In [25]:
# Extraction inventory
inventory_specs = [
    ("FISD rating history", PATHS["fisd_rating_hist"], range(2000, 2025)),
    ("FISD rated issue master", PATHS["fisd_issue_master"], None),
    ("FISD rated issuer master", PATHS["fisd_issuer_master"], None),
    ("FISD issue-issuer map", PATHS["fisd_issue_issuer_map"], None),
    ("FISD extras", PATHS["fisd_extra"], None),
    ("Compustat quarterly", PATHS["compustat_quarterly"], range(1999, 2025)),
    ("Compustat annual", PATHS["compustat_annual"], range(1998, 2025)),
    ("CRSP monthly", PATHS["crsp_monthly"], range(1999, 2025)),
    ("CRSP daily", PATHS["crsp_daily"], range(1999, 2025)),
    ("CRSP market monthly", PATHS["crsp_market_monthly"], range(1999, 2025)),
    ("CRSP market daily", PATHS["crsp_market_daily"], range(1999, 2025)),
    ("CRSP names", PATHS["crsp_names"], None),
    ("CRSP delistings", PATHS["crsp_delistings"], None),
    ("CRSP CCM link", PATHS["crsp_ccm"], None),
    ("Bond-CRSP link", PATHS["bondcrsp_link"], None),
    ("Bond returns std", PATHS["bondret_std"], None),
    ("Bond returns", PATHS["bondret"], None),
    ("IBES", PATHS["ibes"], range(1998, 2025)),
    ("Macro raw directory", PATHS["macro_raw"], None),
    ("Macro treasury rates", PATHS["macro_treasury_rates"], None),
    ("Macro GDP growth", PATHS["macro_gdp_growth"], None),
    ("Macro fed funds", PATHS["macro_fed_funds"], None),
    ("Macro unemployment", PATHS["macro_unemployment"], None),
    ("Macro CPI", PATHS["macro_cpi"], None),
    ("Macro credit spreads", PATHS["macro_credit_spreads"], None),
    ("Macro VIX", PATHS["macro_vix"], None),
]
inventory = pd.DataFrame([path_inventory_row(n, p, list(y) if y else None) for n, p, y in inventory_specs])
display_compact(inventory, n=80)


,dataset,path,exists,parquet_files,total_file_size_mb,manifest_exists,column_audit_exists,expected_years,actual_years,missing_years
0,FISD rating history,data/raw/wrds/fisd_rating_hist,True,25,10.86,False,False,2000-2024,2000-2024,[]
1,FISD rated issue master,data/raw/wrds/fisd_rated_issue_master.parquet,True,1,7.25,True,False,,,[]
2,FISD rated issuer master,data/raw/wrds/fisd_rated_issuer_master.parquet,True,1,0.43,True,False,,,[]
3,FISD issue-issuer map,data/raw/wrds/fisd_issue_issuer_map.parquet,True,1,3.54,True,False,,,[]
4,FISD extras,data/raw/wrds/fisd_extra,True,7,63.09,True,False,,,[]
5,Compustat quarterly,data/raw/wrds/compustat_quarterly,True,26,194.23,False,False,1999-2024,1999-2024,[]
6,Compustat annual,data/raw/wrds/compustat_annual,True,27,91.87,True,True,1998-2024,1998-2024,[]
7,CRSP monthly,data/raw/wrds/crsp/monthly,True,26,88.59,False,False,1999-2024,1999-2024,[]
8,CRSP daily,data/raw/wrds/crsp/daily,True,26,964.65,False,False,1999-2024,1999-2024,[]
9,CRSP market monthly,data/raw/wrds/crsp/market_monthly,True,26,0.20,False,False,1999-2024,1999-2024,[]


In [26]:
# Manifest failure scan
manifest_rows = []
manifest_paths = sorted(WRDS.rglob("extraction_manifest.json"))
if PATHS["macro_manifest"].exists():
    manifest_paths.append(PATHS["macro_manifest"])
for manifest_path in manifest_paths:
    data = read_json(manifest_path)
    if not isinstance(data, dict):
        continue
    for key, rec in data.items():
        if not isinstance(rec, dict):
            continue
        manifest_rows.append({
            "manifest": str(manifest_path.relative_to(ROOT)),
            "dataset": rec.get("dataset", key),
            "partition": key,
            "status": rec.get("status"),
            "row_count": rec.get("row_count", rec.get("total_rows", rec.get("archived_rows"))),
            "min_date": rec.get("min_date", rec.get("actual_min_date", rec.get("overall_min_date"))),
            "max_date": rec.get("max_date", rec.get("actual_max_date", rec.get("overall_max_date"))),
            "error_message": rec.get("error_message"),
            "output_filepath": rec.get("output_filepath"),
        })
manifest_df = pd.DataFrame(manifest_rows)
if manifest_df.empty:
    print("REVIEW: No extraction manifests found under data/raw/wrds/ or data/raw/macro/.")
else:
    bad = manifest_df[manifest_df["status"].isin(["failed", "partial"]) | manifest_df["error_message"].notna()]
    if bad.empty:
        print("PASS: No failed/partial manifest records found.")
    else:
        display_compact(bad, n=50, title="Failed/partial manifest records")
    summary = manifest_df.groupby(["manifest", "status"], dropna=False).size().reset_index(name="records")
    display_compact(summary, n=80, title="Manifest status summary")


PASS: No failed/partial manifest records found.
Manifest status summary


,manifest,status,records
0,data/raw/macro/extraction_manifest.json,success,7
1,data/raw/wrds/bondret/extraction_manifest.json,success,25
2,data/raw/wrds/bondret_std/extraction_manifest....,success,3
3,data/raw/wrds/compustat_annual/extraction_mani...,success,28
4,data/raw/wrds/crsp/extraction_manifest.json,success,107
5,data/raw/wrds/extraction_manifest.json,success,56
6,data/raw/wrds/fisd_extra/extraction_manifest.json,success,7
7,data/raw/wrds/ibes/act_epsus/extraction_manife...,success,29
8,data/raw/wrds/ibes/actu_epsus/extraction_manif...,success,29
9,data/raw/wrds/ibes/recdsum/extraction_manifest...,success,29


In [27]:
# FISD ratings coverage
rating_files = parquet_files(PATHS["fisd_rating_hist"])
rating_summary = partition_summary(rating_files, date_col="rating_date", id_cols=("issue_id",))
display_compact(rating_summary, n=40, title="FISD rating history yearly summary")
ratings = safe_concat(rating_files, columns=["issue_id", "rating_type", "rating_date", "rating", "rating_status", "reason"])
if ratings.empty:
    print("REVIEW: No FISD ratings loaded.")
else:
    ratings["rating_date"] = pd.to_datetime(ratings["rating_date"], errors="coerce")
    ratings["year"] = ratings["rating_date"].dt.year
    display_compact(ratings.groupby("year").agg(rows=("issue_id", "size"), distinct_issues=("issue_id", "nunique")).reset_index(), n=40)
    display_compact(pd.crosstab(ratings["year"], ratings["rating_type"], dropna=False).reset_index(), n=40, title="Rating type by year")
    print("Exact duplicate rows:", duplicate_count(ratings.drop(columns=["__source_file"], errors="ignore")))
    print("Duplicate (issue_id, rating_type, rating_date) rows:", duplicate_count(ratings, ["issue_id", "rating_type", "rating_date"]))
    print("Rating date range:", ratings["rating_date"].min(), "to", ratings["rating_date"].max())
    display_compact(value_counts_table(ratings, "rating", 40), title="Rating values")
    display_compact(value_counts_table(ratings, "rating_status", 30), title="Rating status values")
    reason_cov = ratings["reason"].notna().mean() if "reason" in ratings.columns else np.nan
    print(f"Reason coverage: {reason_cov:.2%}" if pd.notna(reason_cov) else "No reason column")
    agency_spikes = ratings.groupby(["year", "rating_type"]).size().reset_index(name="rows")
    fitch_window = agency_spikes[(agency_spikes["rating_type"].astype(str).str.contains("FR", na=False)) & agency_spikes["year"].between(2014, 2016)]
    if not fitch_window.empty:
        flag("FISD ratings", "REVIEW", "Check Fitch records around 2014-2016", fitch_window.to_dict("records")[:5])
        display_compact(fitch_window, title="Fitch 2014-2016 review window")

FISD rating history yearly summary


,file,year,rows,columns,error,min_date,max_date,distinct_issue_id,duplicate_key_rows
0,fisd_rating_hist_2000.parquet,2000,109711,6,None,2000-01-02,2000-12-29,36745,72966
1,fisd_rating_hist_2001.parquet,2001,73691,6,None,2001-01-01,2001-12-31,28054,45637
2,fisd_rating_hist_2002.parquet,2002,68401,6,None,2002-01-01,2002-12-31,28278,40123
3,fisd_rating_hist_2003.parquet,2003,55528,6,None,2003-01-01,2003-12-31,27061,28467
4,fisd_rating_hist_2004.parquet,2004,46248,6,None,2004-01-01,2004-12-31,24621,21627
5,fisd_rating_hist_2005.parquet,2005,45237,6,None,2005-01-03,2005-12-31,22991,22246
6,fisd_rating_hist_2006.parquet,2006,62045,6,None,2006-01-01,2006-12-29,34435,27610
7,fisd_rating_hist_2007.parquet,2007,62302,6,None,2007-01-02,2007-12-31,35203,27099
8,fisd_rating_hist_2008.parquet,2008,87237,6,None,2008-01-01,2008-12-31,34586,52651
9,fisd_rating_hist_2009.parquet,2009,85926,6,None,2009-01-01,2009-12-31,36248,49678


,year,rows,distinct_issues
0,2000,109711,36745
1,2001,73691,28054
2,2002,68401,28278
3,2003,55528,27061
4,2004,46248,24621
5,2005,45237,22991
6,2006,62045,34435
7,2007,62302,35203
8,2008,87237,34586
9,2009,85926,36248


Rating type by year


rating_type,year,FR,MR,SPR
0,2000,25764,46248,37699
1,2001,17510,31003,25178
2,2002,19777,25547,23077
3,2003,14513,23967,17048
4,2004,14331,19335,12582
5,2005,13810,18695,12732
6,2006,26707,17012,18326
7,2007,20122,24903,17277
8,2008,30510,30272,26455
9,2009,32573,31274,22079


...

,rating,count
0,AAA,383636
1,Aaa,262847
2,A,185084
3,A+,91135
4,AA+,76319
5,BBB+,72542
6,A2,70370
7,AA-,69639
8,BBB,69346
9,A-,66363


Showing 20 of 40 rows
Rating status values


,rating_status,count
0,<NA>,1226537
1,WNOT,321050
2,WOFF,196067
3,WNEG,134415
4,WPOS,32080
5,Off,18373
6,WUND,9240
7,Neg,3104
8,NA,2042
9,Pos,192


Reason coverage: 99.66%
Fitch 2014-2016 review window


,year,rating_type,rows
42,2014,FR,81856
45,2015,FR,101102
48,2016,FR,191029


In [28]:
# FISD master integrity
issue = safe_read(PATHS["fisd_issue_master"])
issuer = safe_read(PATHS["fisd_issuer_master"])
imap = safe_read(PATHS["fisd_issue_issuer_map"])
rows = []
for name, df, key in [("issue_master", issue, "issue_id"), ("issuer_master", issuer, "issuer_id"), ("issue_issuer_map", imap, "issue_id")]:
    rows.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "key": key,
        "key_exists": key in df.columns,
        "null_keys": int(df[key].isna().sum()) if key in df.columns else np.nan,
        "duplicate_keys": duplicate_count(df, [key]) if key in df.columns else np.nan,
        "exact_duplicates": duplicate_count(df),
    })
display_compact(pd.DataFrame(rows), title="FISD master key checks")
if not imap.empty and "issue_id" in imap.columns and "issuer_id" in imap.columns:
    ambig = imap.dropna(subset=["issuer_id"]).groupby("issue_id")["issuer_id"].nunique().reset_index(name="issuer_count")
    display_compact(ambig[ambig["issuer_count"] > 1], title="Issue-to-issuer ambiguity")
if not issue.empty and not imap.empty and "issue_id" in issue.columns and "issue_id" in imap.columns:
    missing_from_issue = set(imap["issue_id"].dropna()) - set(issue["issue_id"].dropna())
    missing_from_map = set(issue["issue_id"].dropna()) - set(imap["issue_id"].dropna())
    print(f"Issues in map absent from issue master: {len(missing_from_issue):,}")
    print(f"Rated issues absent from issue-issuer map: {len(missing_from_map):,}")
if not issue.empty and not issuer.empty and "issuer_id" in issue.columns and "issuer_id" in issuer.columns:
    missing_issuers = set(issue["issuer_id"].dropna()) - set(issuer["issuer_id"].dropna())
    print(f"Issuer IDs present in issues but absent from issuer master: {len(missing_issuers):,}")

FISD master key checks


,dataset,rows,columns,key,key_exists,null_keys,duplicate_keys,exact_duplicates
0,issue_master,298562,23,issue_id,True,0,0,0
1,issuer_master,11907,10,issuer_id,True,0,0,0
2,issue_issuer_map,298562,5,issue_id,True,0,0,0


Issue-to-issuer ambiguity


,issue_id,issuer_count


Issues in map absent from issue master: 0
Rated issues absent from issue-issuer map: 0
Issuer IDs present in issues but absent from issuer master: 1


In [29]:
# FISD date plausibility
candidate = PATHS["fisd_extra"] / "fisd_mergedissue_rated_full.parquet"
issue_dates = safe_read(candidate if candidate.exists() else PATHS["fisd_issue_master"])
if issue_dates.empty:
    print("REVIEW: No rated issue master/full merged issue data available for date plausibility.")
else:
    for c in ["offering_date", "maturity", "as_of_date", "delivery_date", "dated_date"]:
        if c in issue_dates.columns:
            issue_dates[c] = pd.to_datetime(issue_dates[c], errors="coerce")
    checks = []
    if {"offering_date", "maturity"}.issubset(issue_dates.columns):
        checks.append({"check": "maturity before offering_date", "count": int((issue_dates["maturity"] < issue_dates["offering_date"]).sum())})
        checks.append({"check": "offering_date <= maturity valid rows", "count": int((issue_dates["offering_date"] <= issue_dates["maturity"]).sum())})
    for c in ["offering_date", "maturity", "as_of_date"]:
        if c in issue_dates.columns:
            checks.append({"check": f"{c} after 2026-12-31", "count": int((issue_dates[c] > pd.Timestamp("2026-12-31")).sum())})
    if "amount_outstanding" in issue_dates.columns:
        ao = pd.to_numeric(issue_dates["amount_outstanding"], errors="coerce")
        matured = issue_dates.get("maturity", pd.Series(pd.NaT, index=issue_dates.index)) < pd.Timestamp.today()
        live = issue_dates.get("maturity", pd.Series(pd.NaT, index=issue_dates.index)) >= pd.Timestamp.today()
        checks.append({"check": "zero outstanding for matured issues", "count": int(((ao == 0) & matured).sum())})
        checks.append({"check": "zero outstanding for live issues", "count": int(((ao == 0) & live).sum())})
    display_compact(pd.DataFrame(checks), title="Issue date/current amount plausibility")
    if "as_of_date" in issue_dates.columns:
        print("as_of_date coverage:", f"{issue_dates['as_of_date'].notna().mean():.2%}")
md_note = "Note: current amount_outstanding is a current/reference value and must not be used historically without careful timestamp validation."
print(md_note)
flag("FISD", "REVIEW", "Do not use current amount_outstanding historically", md_note)
if not ratings.empty and not issue_dates.empty and "issue_id" in ratings.columns and "offering_date" in issue_dates.columns:
    tmp = ratings[["issue_id", "rating_date"]].merge(issue_dates[["issue_id", "offering_date", "maturity"]].drop_duplicates("issue_id"), on="issue_id", how="left")
    display_compact(pd.DataFrame([
        {"check": "rating_date before offering_date", "count": int((tmp["rating_date"] < tmp["offering_date"]).sum())},
        {"check": "rating_date after maturity", "count": int((tmp["rating_date"] > tmp["maturity"]).sum()) if "maturity" in tmp.columns else np.nan},
    ]))

Issue date/current amount plausibility


,check,count
0,maturity before offering_date,0
1,offering_date <= maturity valid rows,296617
2,offering_date after 2026-12-31,0
3,maturity after 2026-12-31,45719
4,as_of_date after 2026-12-31,1
5,zero outstanding for matured issues,238482
6,zero outstanding for live issues,15195


as_of_date coverage: 1.33%
Note: current amount_outstanding is a current/reference value and must not be used historically without careful timestamp validation.


,check,count
0,rating_date before offering_date,62153
1,rating_date after maturity,2144


In [30]:
# FISD extras inventory
extra_expected = {
    "fisd_rating": "fisd_rating.parquet",
    "fisd_ratings": "fisd_ratings.parquet",
    "fisd_issue_default": "fisd_issue_default.parquet",
    "fisd_issue_affected": "fisd_issue_affected.parquet",
    "fisd_related_issues": "fisd_related_issues.parquet",
    "fisd_issue_enhancement": "fisd_issue_enhancement.parquet",
    "fisd_mergedissue_rated_full": "fisd_mergedissue_rated_full.parquet",
}
rows = []
for name, fname in extra_expected.items():
    p = PATHS["fisd_extra"] / fname
    meta = parquet_metadata(p)
    row = {"dataset": name, "exists": p.exists(), "rows": meta["rows"], "columns": meta["columns"], "error": meta["error"]}
    if p.exists() and meta["error"] is None:
        cols = []
        if name.startswith("fisd_issue") or "mergedissue" in name or "related" in name:
            cols = ["issue_id"]
        df = safe_read(p, columns=cols) if cols else pd.DataFrame()
        if cols and not df.empty:
            row["issue_id_exists"] = "issue_id" in df.columns
            row["duplicate_issue_id"] = duplicate_count(df, ["issue_id"]) if "issue_id" in df.columns else np.nan
    rows.append(row)
display_compact(pd.DataFrame(rows), n=20)

,dataset,exists,rows,columns,error,issue_id_exists,duplicate_issue_id
0,fisd_rating,True,2238858,8,None,NaN,NaN
1,fisd_ratings,True,4417875,8,None,NaN,NaN
2,fisd_issue_default,True,5709,6,None,True,105.0
3,fisd_issue_affected,True,1161,4,None,True,0.0
4,fisd_related_issues,True,31239,7,None,True,5448.0
5,fisd_issue_enhancement,True,156898,8,None,True,2791.0
6,fisd_mergedissue_rated_full,True,298562,224,None,True,0.0


In [31]:
# Compustat quarterly audit
q_files = parquet_files(PATHS["compustat_quarterly"])
q_summary = partition_summary(q_files, date_col="datadate", id_cols=("gvkey",))
_, q_actual, q_missing = expected_missing_years(q_files, 1999, 2024)
print("Missing quarterly years:", q_missing)
display_compact(q_summary, n=40)
q_cols = ["gvkey", "datadate", "indfmt", "datafmt", "popsrc", "consol", "costat", "atq", "ltq", "dlcq", "dlttq", "cheq", "saleq", "oibdpq", "niq", "xintq", "actq", "lctq", "rdq"]
q = safe_concat(q_files, columns=q_cols)
if not q.empty:
    q["datadate"] = pd.to_datetime(q["datadate"], errors="coerce")
    if "rdq" in q.columns:
        q["rdq"] = pd.to_datetime(q["rdq"], errors="coerce")
    print("Exact duplicates:", duplicate_count(q.drop(columns=["__source_file"], errors="ignore")))
    print("Duplicate (gvkey, datadate):", duplicate_count(q, ["gvkey", "datadate"]))
    for c in ["indfmt", "datafmt", "popsrc", "consol", "costat"]:
        display_compact(value_counts_table(q, c), title=f"{c} distribution")
    display_compact(missingness(q, ["atq", "ltq", "dlcq", "dlttq", "cheq", "saleq", "oibdpq", "niq", "xintq", "actq", "lctq", "rdq"]), title="Core quarterly missingness")
    if "rdq" in q.columns:
        print("rdq < datadate rows:", int((q["rdq"] < q["datadate"]).sum()))
        print("missing rdq rows:", int(q["rdq"].isna().sum()))
    format_cols = [c for c in ["indfmt", "datafmt", "popsrc", "consol"] if c in q.columns]
    if format_cols:
        dup_formats = q.groupby(["gvkey", "datadate"])[format_cols].nunique(dropna=False).sum(axis=1).reset_index(name="format_variants")
        print("gvkey-datadate rows with duplicate/multiple format variants:", int((dup_formats["format_variants"] > len(format_cols)).sum()))

Missing quarterly years: []


,file,year,rows,columns,error,min_date,max_date,distinct_gvkey,duplicate_key_rows
0,compustat_quarterly_1999.parquet,1999,51136,63,None,1999-01-31,1999-12-31,13329,37807
1,compustat_quarterly_2000.parquet,2000,49953,63,None,2000-01-31,2000-12-31,13012,36941
2,compustat_quarterly_2001.parquet,2001,47610,63,None,2001-01-31,2001-12-31,12447,35163
3,compustat_quarterly_2002.parquet,2002,45975,63,None,2002-01-31,2002-12-31,11949,34026
4,compustat_quarterly_2003.parquet,2003,45079,63,None,2003-01-31,2003-12-31,11643,33436
5,compustat_quarterly_2004.parquet,2004,44423,63,None,2004-01-31,2004-12-31,11489,32934
6,compustat_quarterly_2005.parquet,2005,44403,63,None,2005-01-31,2005-12-31,11529,32874
7,compustat_quarterly_2006.parquet,2006,44291,63,None,2006-01-31,2006-12-31,11514,32777
8,compustat_quarterly_2007.parquet,2007,44322,63,None,2007-01-31,2007-12-31,11577,32745
9,compustat_quarterly_2008.parquet,2008,43769,63,None,2008-01-31,2008-12-31,11409,32360


Exact duplicates: 311
Duplicate (gvkey, datadate): 990
indfmt distribution


,indfmt,count
0,INDL,1213093


datafmt distribution


...

costat distribution


,costat,count
0,A,663673
1,I,549420


Core quarterly missingness


,column,missing_rate,exists
0,atq,0.250461,True
1,ltq,0.251686,True
2,dlcq,0.291205,True
3,dlttq,0.256543,True
4,cheq,0.253200,True
5,saleq,0.227865,True
6,oibdpq,0.296159,True
7,niq,0.226441,True
8,xintq,0.382969,True
9,actq,0.379162,True


rdq < datadate rows: 94
missing rdq rows: 315580
gvkey-datadate rows with duplicate/multiple format variants: 15


In [32]:
# Compustat annual audit
annual_files = parquet_files(PATHS["compustat_annual"])
a_summary = partition_summary(annual_files, date_col="datadate", id_cols=("gvkey",))
_, a_actual, a_missing = expected_missing_years(annual_files, 1998, 2024)
print("Missing annual years:", a_missing)
display_compact(a_summary, n=40)
a_cols = ["gvkey", "datadate", "indfmt", "datafmt", "popsrc", "consol", "costat", "at", "lt", "dlc", "dltt", "che", "sale", "oibdp", "ni", "xint", "act", "lct"]
a = safe_concat(annual_files, columns=a_cols)
if not a.empty:
    a["datadate"] = pd.to_datetime(a["datadate"], errors="coerce")
    print("Exact duplicates:", duplicate_count(a.drop(columns=["__source_file"], errors="ignore")))
    print("Duplicate (gvkey, datadate):", duplicate_count(a, ["gvkey", "datadate"]))
    for c in ["indfmt", "datafmt", "popsrc", "consol", "costat"]:
        display_compact(value_counts_table(a, c), title=f"Annual {c} distribution")
    display_compact(missingness(a, ["at", "lt", "dlc", "dltt", "che", "sale", "oibdp", "ni", "xint", "act", "lct"]), title="Core annual missingness")
    q_gvkeys = set(q["gvkey"].dropna()) if "q" in globals() and not q.empty and "gvkey" in q.columns else set()
    a_gvkeys = set(a["gvkey"].dropna())
    print(f"Annual gvkeys: {len(a_gvkeys):,}; Quarterly gvkeys: {len(q_gvkeys):,}; Overlap: {len(a_gvkeys & q_gvkeys):,}")

Missing annual years: []


,file,year,rows,columns,error,min_date,max_date,distinct_gvkey,duplicate_key_rows
0,compustat_annual_1998.parquet,1998,21233,68,None,1998-01-31,1998-12-31,12782,8451
1,compustat_annual_1999.parquet,1999,21478,68,None,1999-01-31,1999-12-31,12861,8617
2,compustat_annual_2000.parquet,2000,21306,68,None,2000-01-31,2000-12-31,12527,8779
3,compustat_annual_2001.parquet,2001,20633,68,None,2001-01-31,2001-12-31,11986,8647
4,compustat_annual_2002.parquet,2002,20490,68,None,2002-01-31,2002-12-31,11672,8818
5,compustat_annual_2003.parquet,2003,20016,68,None,2003-01-31,2003-12-31,11467,8549
6,compustat_annual_2004.parquet,2004,19702,68,None,2004-01-31,2004-12-31,11246,8456
7,compustat_annual_2005.parquet,2005,19407,68,None,2005-01-31,2005-12-31,11177,8230
8,compustat_annual_2006.parquet,2006,19412,68,None,2006-01-31,2006-12-31,11115,8297
9,compustat_annual_2007.parquet,2007,19048,68,None,2007-01-31,2007-12-31,11122,7926


Exact duplicates: 0
Duplicate (gvkey, datadate): 219617
Annual indfmt distribution


,indfmt,count
0,INDL,506093
1,FS,32039


Annual datafmt distribution


...

Annual costat distribution


,costat,count
0,A,282161
1,I,255971


Core annual missingness


,column,missing_rate,exists
0,at,0.262187,True
1,lt,0.479408,True
2,dlc,0.479141,True
3,dltt,0.268620,True
4,che,0.538089,True
5,sale,0.308920,True
6,oibdp,0.552340,True
7,ni,0.317142,True
8,xint,0.349013,True
9,act,0.610099,True


Annual gvkeys: 34,114; Quarterly gvkeys: 33,949; Overlap: 33,774


In [33]:
# CRSP monthly audit
m_files = parquet_files(PATHS["crsp_monthly"])
m_summary = partition_summary(m_files, date_col="date", id_cols=("permno", "date"))
_, m_actual, m_missing = expected_missing_years(m_files, 1999, 2024)
print("Missing CRSP monthly years:", m_missing)
display_compact(m_summary, n=40)
m_cols = ["permno", "permco", "date", "ret", "retx", "prc", "shrout", "vol", "cfacpr", "cfacshr"]
monthly_audits = []
for p in m_files:
    df = safe_read(p, columns=m_cols)
    if df.empty:
        continue
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    ret = pd.to_numeric(df.get("ret"), errors="coerce") if "ret" in df.columns else pd.Series(dtype=float)
    prc = pd.to_numeric(df.get("prc"), errors="coerce") if "prc" in df.columns else pd.Series(dtype=float)
    shr = pd.to_numeric(df.get("shrout"), errors="coerce") if "shrout" in df.columns else pd.Series(dtype=float)
    monthly_audits.append({
        "year": year_from_name(p), "rows": len(df), "distinct_permno": df["permno"].nunique() if "permno" in df else np.nan,
        "distinct_permco": df["permco"].nunique() if "permco" in df else np.nan, "min_date": df["date"].min(), "max_date": df["date"].max(),
        "dup_permno_date": duplicate_count(df, ["permno", "date"]), "missing_ret": int(df.get("ret", pd.Series(index=df.index)).isna().sum()),
        "missing_prc": int(df.get("prc", pd.Series(index=df.index)).isna().sum()), "missing_shrout": int(df.get("shrout", pd.Series(index=df.index)).isna().sum()),
        "negative_price": int((prc < 0).sum()), "zero_or_negative_shrout": int((shr <= 0).sum()), "extreme_ret_abs_gt_1": int((ret.abs() > 1).sum()),
    })
display_compact(pd.DataFrame(monthly_audits), n=40)

Missing CRSP monthly years: []


,file,year,rows,columns,error,min_date,max_date,distinct_permno,distinct_date,duplicate_key_rows
0,crsp_monthly_1999.parquet,1999,105292,15,None,1999-01-29,1999-12-31,9604,12,0
1,crsp_monthly_2000.parquet,2000,103202,15,None,2000-01-31,2000-12-29,9323,12,0
2,crsp_monthly_2001.parquet,2001,95981,15,None,2001-01-31,2001-12-31,8609,12,0
3,crsp_monthly_2002.parquet,2002,89551,15,None,2002-01-31,2002-12-31,7934,12,0
4,crsp_monthly_2003.parquet,2003,84797,15,None,2003-01-31,2003-12-31,7514,12,0
5,crsp_monthly_2004.parquet,2004,83607,15,None,2004-01-30,2004-12-31,7376,12,0
6,crsp_monthly_2005.parquet,2005,83764,15,None,2005-01-31,2005-12-30,7401,12,0
7,crsp_monthly_2006.parquet,2006,84065,15,None,2006-01-31,2006-12-29,7496,12,0
8,crsp_monthly_2007.parquet,2007,86068,15,None,2007-01-31,2007-12-31,7738,12,0
9,crsp_monthly_2008.parquet,2008,85049,15,None,2008-01-31,2008-12-31,7444,12,0


,year,rows,distinct_permno,distinct_permco,min_date,max_date,dup_permno_date,missing_ret,missing_prc,missing_shrout,negative_price,zero_or_negative_shrout,extreme_ret_abs_gt_1
0,1999,105292,9604,9421,1999-01-29,1999-12-31,0,4322,3541,707,5564,0,681
1,2000,103202,9323,9100,2000-01-31,2000-12-29,0,4389,3576,717,5123,0,586
2,2001,95981,8609,8380,2001-01-31,2001-12-31,0,3710,3379,299,6165,0,745
3,2002,89551,7934,7705,2002-01-31,2002-12-31,0,3554,3233,289,5425,0,307
4,2003,84797,7514,7297,2003-01-31,2003-12-31,0,3668,3366,267,3841,0,304
5,2004,83607,7376,7140,2004-01-30,2004-12-31,0,3682,3174,426,2547,0,99
6,2005,83764,7401,7119,2005-01-31,2005-12-30,0,3438,2914,459,1957,0,66
7,2006,84065,7496,7059,2006-01-31,2006-12-29,0,3296,2710,556,1797,0,64
8,2007,86068,7738,7089,2007-01-31,2007-12-31,0,3516,2745,669,1680,0,47
9,2008,85049,7444,6642,2008-01-31,2008-12-31,0,2404,2058,299,2516,0,124


In [34]:
# CRSP daily audit
# Processed year-by-year to avoid loading all daily data at once.
d_files = parquet_files(PATHS["crsp_daily"])
daily_rows = []
for p in d_files:
    cols = ["permno", "date", "ret", "retx", "prc", "shrout"]
    df = safe_read(p, columns=cols)
    if df.empty:
        continue
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    ret = pd.to_numeric(df.get("ret"), errors="coerce") if "ret" in df.columns else pd.Series(dtype=float)
    daily_rows.append({
        "year": year_from_name(p), "rows": len(df), "distinct_permno": df["permno"].nunique() if "permno" in df else np.nan,
        "min_date": df["date"].min(), "max_date": df["date"].max(), "dup_permno_date": duplicate_count(df, ["permno", "date"]),
        "missing_ret": int(df.get("ret", pd.Series(index=df.index)).isna().sum()), "missing_prc": int(df.get("prc", pd.Series(index=df.index)).isna().sum()),
        "missing_shrout": int(df.get("shrout", pd.Series(index=df.index)).isna().sum()), "extreme_ret_abs_gt_1": int((ret.abs() > 1).sum()),
    })
daily_summary = pd.DataFrame(daily_rows).sort_values("year") if daily_rows else pd.DataFrame()
display_compact(daily_summary, n=40)
print("Total CRSP daily rows:", f"{daily_summary['rows'].sum():,.0f}" if not daily_summary.empty else "0")

,year,rows,distinct_permno,min_date,max_date,dup_permno_date,missing_ret,missing_prc,missing_shrout,extreme_ret_abs_gt_1
0,1999,2177900,9577,1999-01-04,1999-12-31,0,40114,39319,0,242
1,2000,2134351,9307,2000-01-03,2000-12-29,0,41705,40881,0,167
2,2001,1965766,8595,2001-01-02,2001-12-31,0,46153,45796,0,155
3,2002,1864070,7920,2002-01-02,2002-12-31,0,48858,48524,0,171
4,2003,1766517,7489,2003-01-02,2003-12-31,0,54648,54338,0,87
5,2004,1737946,7352,2004-01-02,2004-12-31,0,50647,50137,0,51
6,2005,1739746,7383,2005-01-03,2005-12-30,0,43369,42839,0,29
7,2006,1736814,7446,2006-01-03,2006-12-29,0,37476,36886,0,33
8,2007,1773601,7712,2007-01-03,2007-12-31,0,33211,32438,0,33
9,2008,1778854,7431,2008-01-02,2008-12-31,0,27252,26900,0,164


Total CRSP daily rows: 49,486,046


In [35]:
# CRSP names, delistings, and CCM
names = safe_read(PATHS["crsp_names"])
delist = safe_read(PATHS["crsp_delistings"])
ccm = safe_read(PATHS["crsp_ccm"])

if not names.empty:
    for c in ["namedt", "nameendt"]:
        if c in names.columns:
            names[c] = pd.to_datetime(names[c], errors="coerce")
    display_compact(pd.DataFrame([{
        "rows": len(names), "distinct_permno": names.get("permno", pd.Series(dtype=object)).nunique(), "missing_permno": names.get("permno", pd.Series(index=names.index)).isna().sum(),
        "inverted_namedt_nameendt": int((names.get("namedt", pd.Series(pd.NaT, index=names.index)) > names.get("nameendt", pd.Series(pd.NaT, index=names.index))).sum()),
        "duplicate_permno_namedt": duplicate_count(names, ["permno", "namedt"]),
    }]), title="CRSP names summary")
    display_compact(value_counts_table(names, "shrcd"), title="shrcd distribution")
    display_compact(value_counts_table(names, "exchcd"), title="exchcd distribution")

if not delist.empty:
    if "dlstdt" in delist.columns:
        delist["dlstdt"] = pd.to_datetime(delist["dlstdt"], errors="coerce")
    display_compact(pd.DataFrame([{
        "rows": len(delist), "distinct_permno": delist.get("permno", pd.Series(dtype=object)).nunique(), "min_dlstdt": delist.get("dlstdt", pd.Series(dtype="datetime64[ns]")).min(),
        "max_dlstdt": delist.get("dlstdt", pd.Series(dtype="datetime64[ns]")).max(), "dup_permno_dlstdt": duplicate_count(delist, ["permno", "dlstdt"]),
        "missing_dlret": delist.get("dlret", pd.Series(index=delist.index)).isna().sum() if "dlret" in delist.columns else np.nan,
    }]), title="CRSP delistings summary")
    display_compact(value_counts_table(delist, "dlstcd"), title="dlstcd distribution")

if not ccm.empty:
    for c in ["linkdt", "linkenddt"]:
        if c in ccm.columns:
            ccm[c] = pd.to_datetime(ccm[c], errors="coerce")
    required = ["gvkey", "lpermno", "linktype", "linkprim", "linkdt", "linkenddt"]
    display_compact(pd.DataFrame([{
        "rows": len(ccm), **{f"has_{c}": c in ccm.columns for c in required},
        "inverted_link_ranges": int((ccm.get("linkdt", pd.Series(pd.NaT, index=ccm.index)) > ccm.get("linkenddt", pd.Series(pd.NaT, index=ccm.index))).sum()),
        "exact_duplicates": duplicate_count(ccm), "duplicate_records": duplicate_count(ccm, [c for c in required if c in ccm.columns]),
    }]), title="CCM summary")
    display_compact(value_counts_table(ccm, "linktype"), title="linktype distribution")
    display_compact(value_counts_table(ccm, "linkprim"), title="linkprim distribution")
    # Simple overlap screen by gvkey/permno with date ranges; not a filter.
    overlap_rows = []
    for key in ["gvkey", "lpermno"]:
        if {key, "linkdt", "linkenddt"}.issubset(ccm.columns):
            tmp = ccm.sort_values([key, "linkdt"])
            tmp["prev_end"] = tmp.groupby(key)["linkenddt"].shift()
            overlap_rows.append({"key": key, "overlapping_adjacent_ranges": int((tmp["linkdt"] <= tmp["prev_end"]).sum())})
    display_compact(pd.DataFrame(overlap_rows), title="CCM adjacent overlap screen")

CRSP names summary


,rows,distinct_permno,missing_permno,inverted_namedt_nameendt,duplicate_permno_namedt
0,117830,38843,0,0,0


shrcd distribution


,shrcd,count
0,11,63685
1,10,20130
2,73,12736
3,12,7876
4,31,4388
5,14,2560
6,44,2041
7,18,1696
8,71,880
9,48,747


exchcd distribution


...

,linktype,count
0,NR,27427
1,NU,22764
2,LC,17932
3,LU,15945
4,LS,7159
5,LX,1176
6,LN,186
7,LD,119
8,NP,3


linkprim distribution


,linkprim,count
0,P,43593
1,C,43569
2,N,3604
3,J,1945


CCM adjacent overlap screen


,key,overlapping_adjacent_ranges
0,gvkey,4161
1,lpermno,907


In [36]:
# CRSP market indexes
for label, path, expected in [("market_monthly", PATHS["crsp_market_monthly"], range(1999, 2025)), ("market_daily", PATHS["crsp_market_daily"], range(1999, 2025))]:
    files = parquet_files(path)
    _, actual, missing = expected_missing_years(files, min(expected), max(expected)) if files else ([], [], list(expected))
    print(f"{label}: missing years", missing[:20])
    rows = []
    for p in files:
        df = safe_read(p, columns=["date", "vwretd", "ewretd", "sprtrn"])
        if df.empty:
            continue
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        rows.append({"year": year_from_name(p), "rows": len(df), "min_date": df["date"].min(), "max_date": df["date"].max(), "duplicate_dates": duplicate_count(df, ["date"]), **{f"missing_{c}": int(df[c].isna().sum()) for c in ["vwretd", "ewretd", "sprtrn"] if c in df.columns}})
    display_compact(pd.DataFrame(rows).sort_values("year") if rows else pd.DataFrame(), n=40, title=label)

market_monthly: missing years []
market_monthly


,year,rows,min_date,max_date,duplicate_dates,missing_vwretd,missing_ewretd,missing_sprtrn
0,1999,12,1999-01-29,1999-12-31,0,0,0,0
1,2000,12,2000-01-31,2000-12-29,0,0,0,0
2,2001,12,2001-01-31,2001-12-31,0,0,0,0
3,2002,12,2002-01-31,2002-12-31,0,0,0,0
4,2003,12,2003-01-31,2003-12-31,0,0,0,0
5,2004,12,2004-01-30,2004-12-31,0,0,0,0
6,2005,12,2005-01-31,2005-12-30,0,0,0,0
7,2006,12,2006-01-31,2006-12-29,0,0,0,0
8,2007,12,2007-01-31,2007-12-31,0,0,0,0
9,2008,12,2008-01-31,2008-12-31,0,0,0,0


market_daily: missing years []
market_daily


,year,rows,min_date,max_date,duplicate_dates,missing_vwretd,missing_ewretd,missing_sprtrn
0,1999,252,1999-01-04,1999-12-31,0,0,0,0
1,2000,252,2000-01-03,2000-12-29,0,0,0,0
2,2001,248,2001-01-02,2001-12-31,0,0,0,0
3,2002,252,2002-01-02,2002-12-31,0,0,0,0
4,2003,252,2003-01-02,2003-12-31,0,0,0,0
5,2004,252,2004-01-02,2004-12-31,0,0,0,0
6,2005,252,2005-01-03,2005-12-30,0,0,0,0
7,2006,251,2006-01-03,2006-12-29,0,0,0,0
8,2007,251,2007-01-03,2007-12-31,0,0,0,0
9,2008,253,2008-01-02,2008-12-31,0,0,0,0


In [37]:
# Bond-CRSP link audit
bcl = safe_read(PATHS["bondcrsp_link"])
if bcl.empty:
    print("REVIEW: Bond-CRSP link file not available.")
else:
    date_cols = [c for c in bcl.columns if "date" in c.lower() or c.lower().endswith("dt")]
    for c in date_cols:
        bcl[c] = pd.to_datetime(bcl[c], errors="coerce")
    cusip_col = next((c for c in ["issue_cusip", "cusip", "bond_cusip"] if c in bcl.columns), None)
    permno_col = next((c for c in ["permno", "lpermno"] if c in bcl.columns), None)
    summary = {"rows": len(bcl), "exact_duplicates": duplicate_count(bcl)}
    if cusip_col: summary.update({"distinct_cusip": bcl[cusip_col].nunique(), "null_cusip": int(bcl[cusip_col].isna().sum())})
    if permno_col: summary.update({"distinct_permno": bcl[permno_col].nunique(), "null_permno": int(bcl[permno_col].isna().sum())})
    display_compact(pd.DataFrame([summary]))
    if len(date_cols) >= 2:
        display_compact(pd.DataFrame([{"date_col_1": date_cols[0], "date_col_2": date_cols[1], "inverted_ranges": int((bcl[date_cols[0]] > bcl[date_cols[1]]).sum())}]))
    if date_cols:
        year_cov = pd.to_datetime(bcl[date_cols[0]], errors="coerce").dt.year.value_counts().sort_index().reset_index()
        year_cov.columns = ["year", "rows"]
        display_compact(year_cov, n=40, title="Bond-CRSP link coverage by year")

,rows,exact_duplicates,distinct_cusip,null_cusip,distinct_permno,null_permno
0,358312,0,351717,0,4145,0


,date_col_1,date_col_2,inverted_ranges
0,trace_startdt,trace_enddt,0


Bond-CRSP link coverage by year


,year,rows
0,2002,12544
1,2003,5506
2,2004,4641
3,2005,3374
4,2006,2715
5,2007,3413
6,2008,3851
7,2009,4191
8,2010,6747
9,2011,7769


In [38]:
# Bond returns audit
for label in ["bondret_std", "bondret"]:
    files = parquet_files(PATHS[label])
    if not files:
        print(f"{label}: not present")
        continue
    manifest = read_json(PATHS[label] / "extraction_manifest.json") or {}
    date_col = None
    for rec in manifest.values():
        if isinstance(rec, dict) and rec.get("date_column"):
            date_col = rec["date_column"]; break
    rows = []
    for p in files:
        meta = parquet_metadata(p)
        cols = None
        df = safe_read(p) if meta["rows"] and meta["rows"] < 2_000_000 else pd.DataFrame()
        row = {"file": p.name, "year": year_from_name(p), "rows": meta["rows"], "columns": meta["columns"], "error": meta["error"]}
        if not df.empty:
            dc = date_col if date_col in df.columns else next((c for c in df.columns if "date" in c.lower()), None)
            if dc:
                df[dc] = pd.to_datetime(df[dc], errors="coerce")
                row.update({"min_date": df[dc].min(), "max_date": df[dc].max()})
            id_cols = [c for c in ["issue_id", "cusip", "issue_cusip", "bond_sym_id"] if c in df.columns]
            for c in id_cols[:2]: row[f"distinct_{c}"] = df[c].nunique(dropna=True)
            for c in ["yield", "price_eom", "ret_eom", "duration", "rating", "rating_num"]:
                if c in df.columns: row[f"missing_{c}"] = int(df[c].isna().sum())
        rows.append(row)
    out = pd.DataFrame(rows).sort_values("year") if rows else pd.DataFrame()
    display_compact(out, n=80, title=label)
    if not out.empty and out["year"].dropna().min() >= 2025:
        print(f"NOTE: {label} coverage appears 2025+ and post-sample only.")

bondret_std


,file,year,rows,columns,error,min_date,max_date,distinct_issue_id,distinct_cusip,missing_yield,missing_price_eom,missing_ret_eom,missing_duration,missing_rating_num
0,bondret_std_2025.parquet,2025,164797,59,None,2025-04-30,2025-12-31,31254,31254,1774,0,2884,2108,53850
1,bondret_std_2026.parquet,2026,14425,59,None,2026-01-31,2026-01-31,14425,14425,116,0,132,145,3695


NOTE: bondret_std coverage appears 2025+ and post-sample only.
bondret


,file,year,rows,columns,error,min_date,max_date,distinct_issue_id,distinct_cusip,missing_yield,missing_price_eom,missing_ret_eom,missing_duration,missing_rating_num
0,bondret_2002.parquet,2002,47995,59,None,2002-07-31,2002-12-31,9212,9212,502,0,9210,882,28084
1,bondret_2003.parquet,2003,99451,59,None,2003-01-31,2003-12-31,9937,9937,1123,0,1716,2572,26423
2,bondret_2004.parquet,2004,94679,59,None,2004-01-31,2004-12-31,9168,9168,887,0,1245,2309,17124
3,bondret_2005.parquet,2005,91734,59,None,2005-01-31,2005-12-31,8807,8807,849,0,1095,1691,15005
4,bondret_2006.parquet,2006,90017,59,None,2006-01-31,2006-12-31,8544,8544,817,0,1140,1269,11399
5,bondret_2007.parquet,2007,90693,59,None,2007-01-31,2007-12-31,8800,8800,879,0,1392,1320,8314
6,bondret_2008.parquet,2008,95201,59,None,2008-01-31,2008-12-31,9788,9788,961,0,2244,1346,10084
7,bondret_2009.parquet,2009,101980,59,None,2009-01-31,2009-12-31,10496,10496,1335,0,2299,1609,13739
8,bondret_2010.parquet,2010,112919,59,None,2010-01-31,2010-12-31,12325,12325,1273,0,3825,1812,26555
9,bondret_2011.parquet,2011,124924,59,None,2011-01-31,2011-12-31,13710,13710,1506,0,4050,2164,37189


In [39]:
# IBES audit
# Table-specific schemas are intentional:
# - statsum_epsus contains consensus forecast-period fields.
# - act_epsus / actu_epsus contain realized actual EPS fields.
PROJECT_ROOT = ROOT
ibes_root = PATHS["ibes"]
ibes_tables = ["statsum_epsus", "act_epsus", "actu_epsus", "recdsum"]

CONSENSUS_COLUMNS = [
    "ticker", "cusip", "oftic", "cname", "statpers", "fpedats", "fpi",
    "measure", "numest", "meanest", "medest", "stdev",
]
ACTUALS_COLUMNS = [
    "ticker", "cusip", "oftic", "cname", "pends", "measure", "pdicity",
    "anndats", "actdats", "value", "curr_act", "usfirm",
]
RECSUM_COLUMNS = [
    "ticker", "cusip", "oftic", "cname", "statpers", "anndats",
    "ireccd", "estrec", "meanrec", "medianrec", "numrec",
]
TABLE_COLUMNS = {
    "statsum_epsus": CONSENSUS_COLUMNS,
    "act_epsus": ACTUALS_COLUMNS,
    "actu_epsus": ACTUALS_COLUMNS,
    "recdsum": RECSUM_COLUMNS,
}


def read_existing_columns(path, requested):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame(), [], requested
    try:
        available = pq.read_schema(path).names if pq is not None else pd.read_parquet(path).columns.tolist()
        selected = [c for c in requested if c in available]
        missing = [c for c in requested if c not in available]
        df = pd.read_parquet(path, columns=selected) if selected else pd.DataFrame()
        return df, selected, missing
    except Exception as exc:
        flag(path.name, "FAIL", "IBES schema/read error", str(exc))
        return pd.DataFrame(), [], requested

# Explicit schema sanity check requested for actuals.
act_2024_path = PROJECT_ROOT / "data" / "raw" / "wrds" / "ibes" / "act_epsus" / "act_epsus_2024.parquet"
if act_2024_path.exists():
    df_act_sample, selected, missing = read_existing_columns(act_2024_path, ACTUALS_COLUMNS)
    print("act_epsus_2024 selected:", selected)
    print("act_epsus_2024 missing:", missing)
    display(df_act_sample.head())
else:
    print(f"act_epsus_2024 not present: {act_2024_path}")

for table in ibes_tables:
    tdir = ibes_root / table
    files = [p for p in parquet_files(tdir) if "null_" not in p.stem]
    manifest = read_json(tdir / "extraction_manifest.json") or {}
    summary = manifest.get(f"{table}_summary", {}) if isinstance(manifest, dict) else {}
    date_col = summary.get("date_column")
    _, actual, missing_years = expected_missing_years(files, 1998, 2024) if files else ([], [], list(range(1998, 2025)))
    print(
        f"\n{table}: date_col={date_col}; missing_years={missing_years[:20]}; "
        f"extracted_row_difference={summary.get('extracted_row_difference')}; "
        f"null_date_rows={summary.get('null_date_rows')}"
    )
    requested = list(dict.fromkeys([date_col] + TABLE_COLUMNS.get(table, []) if date_col else TABLE_COLUMNS.get(table, [])))
    rows = []
    missing_by_file = []
    for p in files:
        df, selected, missing_cols = read_existing_columns(p, requested)
        if missing_cols:
            missing_by_file.append({"file": p.name, "missing_columns": missing_cols})
        if df.empty:
            continue
        if date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

        if table == "statsum_epsus":
            key_cols = [c for c in ["ticker", "fpedats", "fpi", "measure", date_col] if c in df.columns]
            estimate_missing = {
                "missing_meanest": int(df["meanest"].isna().sum()) if "meanest" in df.columns else np.nan,
                "missing_stdev": int(df["stdev"].isna().sum()) if "stdev" in df.columns else np.nan,
                "missing_numest": int(df["numest"].isna().sum()) if "numest" in df.columns else np.nan,
            }
        elif table in {"act_epsus", "actu_epsus"}:
            key_cols = [c for c in ["ticker", "pends", "measure", date_col] if c in df.columns]
            estimate_missing = {
                "missing_value": int(df["value"].isna().sum()) if "value" in df.columns else np.nan,
                "missing_pends": int(df["pends"].isna().sum()) if "pends" in df.columns else np.nan,
            }
        else:
            key_cols = [c for c in ["ticker", "measure", date_col] if c in df.columns]
            estimate_missing = {
                "missing_meanrec": int(df["meanrec"].isna().sum()) if "meanrec" in df.columns else np.nan,
                "missing_numrec": int(df["numrec"].isna().sum()) if "numrec" in df.columns else np.nan,
            }

        row = {
            "year": year_from_name(p),
            "rows": len(df),
            "min_date": df[date_col].min() if date_col in df.columns else pd.NaT,
            "max_date": df[date_col].max() if date_col in df.columns else pd.NaT,
            "duplicate_inferred_key": duplicate_count(df, key_cols) if key_cols else np.nan,
            "missing_ticker": int(df["ticker"].isna().sum()) if "ticker" in df.columns else np.nan,
            "missing_cusip": int(df["cusip"].isna().sum()) if "cusip" in df.columns else np.nan,
            **estimate_missing,
        }
        rows.append(row)
    display_compact(pd.DataFrame(rows).sort_values("year") if rows else pd.DataFrame(), n=40)
    if missing_by_file:
        display_compact(pd.DataFrame(missing_by_file), n=10, title=f"{table} requested columns absent from schema")

    sample_files = files[:2]
    sample_frames = []
    for p in sample_files:
        df, _, _ = read_existing_columns(p, requested)
        if not df.empty:
            sample_frames.append(df)
    sample = pd.concat(sample_frames, ignore_index=True) if sample_frames else pd.DataFrame()
    if not sample.empty:
        if table == "statsum_epsus":
            display_compact(value_counts_table(sample, "fpi"), title=f"{table} fpi distribution sample")
            for c in ["numest", "stdev"]:
                if c in sample.columns:
                    print(f"{table} {c} coverage sample: {sample[c].notna().mean():.2%}")
        elif table in {"act_epsus", "actu_epsus"}:
            display_compact(value_counts_table(sample, "measure"), title=f"{table} measure distribution sample")
            if "value" in sample.columns:
                print(f"{table} value coverage sample: {sample['value'].notna().mean():.2%}")
        else:
            for c in ["meanrec", "numrec"]:
                if c in sample.columns:
                    print(f"{table} {c} coverage sample: {sample[c].notna().mean():.2%}")


act_epsus_2024 selected: ['ticker', 'cusip', 'oftic', 'cname', 'pends', 'measure', 'pdicity', 'anndats', 'actdats', 'value', 'curr_act', 'usfirm']
act_epsus_2024 missing: []


,ticker,cusip,oftic,cname,pends,measure,pdicity,anndats,actdats,value,curr_act,usfirm
0,ALUM,09999999,AL.CP,ALUMINUM,2023-12-31,EPS,ANN,2024-01-01,2024-01-04,1.02,USD,1
1,OILP,OILPOILP,OIL.CP,"OIL, CRUDE, WTI",2023-12-31,EPS,QTR,2024-01-01,2024-01-04,78.64,USD,1
2,PALL,PALLPALL,PALL.C,PALLADIUM (PER T,2023-12-31,EPS,ANN,2024-01-01,2024-01-04,1340.48,USD,1
3,PALL,PALLPALL,PALL.C,PALLADIUM (PER T,2023-12-31,EPS,QTR,2024-01-01,2024-01-04,1096.42,USD,1
4,PLTN,PLATINUM,PLTN.C,PLATINUM (PER TR,2023-12-31,EPS,ANN,2024-01-01,2024-01-04,966.85,USD,1



statsum_epsus: date_col=statpers; missing_years=[]; extracted_row_difference=0; null_date_rows=0


,year,rows,min_date,max_date,duplicate_inferred_key,missing_ticker,missing_cusip,missing_meanest,missing_stdev,missing_numest
0,1998,414649,1998-01-15,1998-12-17,30,0,6,53,125065,0
1,1999,403661,1999-01-14,1999-12-16,20,0,0,72,119266,0
2,2000,382016,2000-01-20,2000-12-14,15,0,0,68,114035,0
3,2001,345066,2001-01-18,2001-12-20,12,0,0,71,101931,0
4,2002,320965,2002-01-17,2002-12-19,1,0,0,8,94458,0
5,2003,330300,2003-01-16,2003-12-18,0,0,1,0,97397,0
6,2004,357910,2004-01-15,2004-12-16,0,0,13,14,100867,0
7,2005,378576,2005-01-20,2005-12-15,29,0,0,67,101160,0
8,2006,391624,2006-01-19,2006-12-14,176,0,0,72,101858,0
9,2007,407378,2007-01-18,2007-12-20,485,0,0,155,103979,0


statsum_epsus fpi distribution sample


...

actu_epsus value coverage sample: 94.57%

recdsum: date_col=statpers; missing_years=[]; extracted_row_difference=0; null_date_rows=0


,year,rows,min_date,max_date,duplicate_inferred_key,missing_ticker,missing_cusip,missing_meanrec,missing_numrec
0,1998,188909,1998-01-15,1998-12-17,0,0,81,0,0
1,1999,191682,1999-01-14,1999-12-16,0,0,256,0,0
2,2000,193436,2000-01-20,2000-12-14,0,0,303,0,0
3,2001,188605,2001-01-18,2001-12-20,0,0,304,0,0
4,2002,174717,2002-01-17,2002-12-19,0,0,362,0,0
5,2003,158573,2003-01-16,2003-12-18,0,0,627,0,0
6,2004,157189,2004-01-15,2004-12-16,0,0,605,0,0
7,2005,175296,2005-01-20,2005-12-15,0,0,57,0,0
8,2006,194493,2006-01-19,2006-12-14,0,0,33,0,0
9,2007,201801,2007-01-18,2007-12-20,0,0,21,0,0


recdsum requested columns absent from schema


,file,missing_columns
0,recdsum_1998.parquet,"[anndats, ireccd, estrec, medianrec]"
1,recdsum_1999.parquet,"[anndats, ireccd, estrec, medianrec]"
2,recdsum_2000.parquet,"[anndats, ireccd, estrec, medianrec]"
3,recdsum_2001.parquet,"[anndats, ireccd, estrec, medianrec]"
4,recdsum_2002.parquet,"[anndats, ireccd, estrec, medianrec]"
5,recdsum_2003.parquet,"[anndats, ireccd, estrec, medianrec]"
6,recdsum_2004.parquet,"[anndats, ireccd, estrec, medianrec]"
7,recdsum_2005.parquet,"[anndats, ireccd, estrec, medianrec]"
8,recdsum_2006.parquet,"[anndats, ireccd, estrec, medianrec]"
9,recdsum_2007.parquet,"[anndats, ireccd, estrec, medianrec]"


Showing 10 of 27 rows
recdsum meanrec coverage sample: 100.00%
recdsum numrec coverage sample: 100.00%


In [40]:
# Macro/FRED audit
# Current raw macro extractor writes separate native-frequency files under data/raw/macro/.
# HY OAS and FRED S&P 500 are intentionally not required here; CRSP market indexes are audited separately.
macro_manifest = read_json(PATHS["macro_manifest"]) or {}
macro_rows = []
for dataset, spec in MACRO_DATASETS.items():
    p = spec["path"]
    rec = macro_manifest.get(dataset, {}) if isinstance(macro_manifest, dict) else {}
    row = {
        "dataset": dataset,
        "path": str(p.relative_to(ROOT)) if p.exists() else str(p),
        "exists": p.exists(),
        "manifest_status": rec.get("status"),
        "fred_series_ids": ",".join(spec["series_ids"]),
        "expected_columns": spec["columns"],
        "missing_expected_columns": None,
        "rows": np.nan,
        "columns": np.nan,
        "date_index_name": None,
        "min_date": rec.get("actual_min_date"),
        "max_date": rec.get("actual_max_date"),
        "duplicate_dates": np.nan,
        "missing_observations": np.nan,
        "file_size_mb": round(file_size_mb(p), 3) if p.exists() else 0,
    }
    if p.exists():
        try:
            df = pd.read_parquet(p)
            row["rows"] = len(df)
            row["columns"] = len(df.columns)
            row["date_index_name"] = df.index.name
            missing_cols = [c for c in spec["columns"] if c not in df.columns]
            row["missing_expected_columns"] = missing_cols
            idx = pd.to_datetime(df.index, errors="coerce")
            if len(idx):
                row["min_date"] = idx.min()
                row["max_date"] = idx.max()
                row["duplicate_dates"] = int(pd.Index(idx).duplicated().sum())
            row["missing_observations"] = int(df.isna().sum().sum())
            if df.index.name != "date":
                flag("macro", "FAIL", f"{dataset} index is not named date", str(df.index.name))
            if missing_cols:
                flag("macro", "FAIL", f"{dataset} missing expected columns", str(missing_cols))
            if idx.max() is not pd.NaT and pd.notna(idx.max()) and idx.max() > pd.Timestamp("2024-12-31"):
                flag("macro", "REVIEW", f"{dataset} has observations after 2024", str(idx.max()))
        except Exception as exc:
            row["read_error"] = str(exc)
            flag("macro", "FAIL", f"{dataset} read error", str(exc))
    else:
        flag("macro", "REVIEW", f"Missing macro raw file: {dataset}", str(p))
    macro_rows.append(row)

macro_summary = pd.DataFrame(macro_rows)
display_compact(macro_summary, n=20)

if macro_manifest:
    manifest_compact = pd.DataFrame([
        {
            "dataset": k,
            "status": v.get("status"),
            "fred_series_ids": v.get("fred_series_ids"),
            "requested_start_date": v.get("requested_start_date"),
            "requested_end_date": v.get("requested_end_date"),
            "actual_min_date": v.get("actual_min_date"),
            "actual_max_date": v.get("actual_max_date"),
            "row_count": v.get("row_count"),
            "missing_value_counts": v.get("missing_value_counts"),
            "error_message": v.get("error_message"),
        }
        for k, v in macro_manifest.items()
        if isinstance(v, dict)
    ])
    display_compact(manifest_compact, n=20, title="Macro extraction manifest")
else:
    print("REVIEW: Macro extraction manifest not found at data/raw/macro/extraction_manifest.json")

# Explicit project decisions: these are not raw macro requirements.
print("NOTE: HY OAS / ICE OAS / HYG / JNK are intentionally excluded from raw macro extraction.")
print("NOTE: FRED S&P 500 is intentionally excluded; CRSP market-index returns are canonical.")


,dataset,path,exists,manifest_status,fred_series_ids,expected_columns,missing_expected_columns,rows,columns,date_index_name,min_date,max_date,duplicate_dates,missing_observations,file_size_mb
0,treasury_rates,data/raw/macro/treasury_rates.parquet,True,success,"DGS3MO,DGS10","[treasury_3m, treasury_10y, term_spread]",[],6522,3,date,2000-01-03,2024-12-31,0,807,0.098
1,gdp_growth,data/raw/macro/gdp_growth.parquet,True,success,A191RL1Q225SBEA,[gdp_growth_qoq],[],100,1,date,2000-01-01,2024-10-01,0,0,0.003
2,fed_funds,data/raw/macro/fed_funds.parquet,True,success,FEDFUNDS,[fed_funds_rate],[],300,1,date,2000-01-01,2024-12-01,0,0,0.006
3,unemployment,data/raw/macro/unemployment.parquet,True,success,UNRATE,"[unemployment_rate, unemployment_change]",[],300,2,date,2000-01-01,2024-12-01,0,1,0.007
4,cpi,data/raw/macro/cpi.parquet,True,success,CPIAUCSL,"[cpi, cpi_yoy]",[],300,2,date,2000-01-01,2024-12-01,0,12,0.010
5,credit_spreads,data/raw/macro/credit_spreads.parquet,True,success,"AAA10Y,BAA10Y","[aaa10y, baa10y]",[],6522,2,date,2000-01-03,2024-12-31,0,548,0.078
6,vix,data/raw/macro/vix.parquet,True,success,VIXCLS,[vix],[],6522,1,date,2000-01-03,2024-12-31,0,213,0.081


Macro extraction manifest


,dataset,status,fred_series_ids,requested_start_date,requested_end_date,actual_min_date,actual_max_date,row_count,missing_value_counts,error_message
0,cpi,success,[CPIAUCSL],2000-01-01,2024-12-31,2000-01-01,2024-12-01,300,"{'cpi': 0, 'cpi_yoy': 12}",None
1,credit_spreads,success,"[AAA10Y, BAA10Y]",2000-01-01,2024-12-31,2000-01-03,2024-12-31,6522,"{'aaa10y': 274, 'baa10y': 274}",None
2,fed_funds,success,[FEDFUNDS],2000-01-01,2024-12-31,2000-01-01,2024-12-01,300,{'fed_funds_rate': 0},None
3,gdp_growth,success,[A191RL1Q225SBEA],2000-01-01,2024-12-31,2000-01-01,2024-10-01,100,{'gdp_growth_qoq': 0},None
4,treasury_rates,success,"[DGS3MO, DGS10]",2000-01-01,2024-12-31,2000-01-03,2024-12-31,6522,"{'term_spread': 269, 'treasury_10y': 269, 'tre...",None
5,unemployment,success,[UNRATE],2000-01-01,2024-12-31,2000-01-01,2024-12-01,300,"{'unemployment_change': 1, 'unemployment_rate'...",None
6,vix,success,[VIXCLS],2000-01-01,2024-12-31,2000-01-03,2024-12-31,6522,{'vix': 213},None


NOTE: HY OAS / ICE OAS / HYG / JNK are intentionally excluded from raw macro extraction.
NOTE: FRED S&P 500 is intentionally excluded; CRSP market-index returns are canonical.


In [41]:
# Identifier overlap and mergeability
sets = {}
sets["fisd_issue_ids"] = set(issue["issue_id"].dropna()) if "issue" in globals() and "issue_id" in issue.columns else set()
sets["fisd_issuer_ids"] = set(issuer["issuer_id"].dropna()) if "issuer" in globals() and "issuer_id" in issuer.columns else set()
sets["compustat_gvkeys"] = (set(q["gvkey"].dropna()) if "q" in globals() and "gvkey" in q.columns else set()) | (set(a["gvkey"].dropna()) if "a" in globals() and "gvkey" in a.columns else set())
sets["ccm_gvkeys"] = set(ccm["gvkey"].dropna()) if "ccm" in globals() and "gvkey" in ccm.columns else set()
sets["ccm_permnos"] = set(ccm["lpermno"].dropna()) if "ccm" in globals() and "lpermno" in ccm.columns else set()
monthly_permnos = set()
for p in parquet_files(PATHS["crsp_monthly"]):
    df = safe_read(p, columns=["permno"])
    if "permno" in df.columns: monthly_permnos.update(df["permno"].dropna().unique())
sets["crsp_monthly_permnos"] = monthly_permnos
bcl_cusips = set()
if "bcl" in globals() and not bcl.empty:
    ccol = next((c for c in ["issue_cusip", "cusip", "bond_cusip"] if c in bcl.columns), None)
    if ccol: bcl_cusips = set(bcl[ccol].dropna())
sets["bondcrsp_cusips"] = bcl_cusips
ibes_ids = set()
for table in ["statsum_epsus", "act_epsus", "actu_epsus", "recdsum"]:
    for p in parquet_files(PATHS["ibes"] / table)[:3]:
        df = safe_read(p, columns=[c for c in ["ticker", "cusip", "oftic"] if True])
        for c in ["ticker", "cusip", "oftic"]:
            if c in df.columns: ibes_ids.update(df[c].dropna().astype(str).unique())
sets["ibes_identifiers_sample"] = ibes_ids

overlap_rows = [
    {"metric": "FISD issues", "count": len(sets["fisd_issue_ids"])},
    {"metric": "FISD issuers", "count": len(sets["fisd_issuer_ids"])},
    {"metric": "Compustat gvkeys", "count": len(sets["compustat_gvkeys"])},
    {"metric": "CCM-linked gvkeys", "count": len(sets["ccm_gvkeys"])},
    {"metric": "CCM-linked permnos", "count": len(sets["ccm_permnos"])},
    {"metric": "CRSP monthly permnos", "count": len(sets["crsp_monthly_permnos"])},
    {"metric": "Bond-CRSP linked CUSIPs", "count": len(sets["bondcrsp_cusips"])},
    {"metric": "IBES identifiers sample", "count": len(sets["ibes_identifiers_sample"])},
    {"metric": "Compustat gvkeys with CCM links", "count": len(sets["compustat_gvkeys"] & sets["ccm_gvkeys"])},
    {"metric": "CCM permnos found in CRSP monthly", "count": len(sets["ccm_permnos"] & sets["crsp_monthly_permnos"])},
]
display_compact(pd.DataFrame(overlap_rows), n=30)
print("FISD issuer-to-Compustat linkage is not fully available in current raw files; no mapping is invented here.")

,metric,count
0,FISD issues,298562
1,FISD issuers,11907
2,Compustat gvkeys,34289
3,CCM-linked gvkeys,36147
4,CCM-linked permnos,36938
5,CRSP monthly permnos,24701
6,Bond-CRSP linked CUSIPs,351717
7,IBES identifiers sample,53667
8,Compustat gvkeys with CCM links,24607
9,CCM permnos found in CRSP monthly,24627


FISD issuer-to-Compustat linkage is not fully available in current raw files; no mapping is invented here.


In [42]:
# Merge attrition table
stages = [
    {"stage": "FISD rated issues", "distinct_issues": len(sets.get("fisd_issue_ids", set())), "distinct_issuers": np.nan, "distinct_gvkeys": np.nan, "distinct_permnos": np.nan},
    {"stage": "FISD rated issuers", "distinct_issues": np.nan, "distinct_issuers": len(sets.get("fisd_issuer_ids", set())), "distinct_gvkeys": np.nan, "distinct_permnos": np.nan},
    {"stage": "issue-issuer mapping", "distinct_issues": imap["issue_id"].nunique() if "imap" in globals() and "issue_id" in imap.columns else np.nan, "distinct_issuers": imap["issuer_id"].nunique() if "imap" in globals() and "issuer_id" in imap.columns else np.nan, "distinct_gvkeys": np.nan, "distinct_permnos": np.nan},
    {"stage": "bond-CRSP matched", "distinct_issues": np.nan, "distinct_issuers": np.nan, "distinct_gvkeys": np.nan, "distinct_permnos": len(sets.get("bondcrsp_cusips", set()))},
    {"stage": "Compustat matched", "distinct_issues": np.nan, "distinct_issuers": np.nan, "distinct_gvkeys": len(sets.get("compustat_gvkeys", set())), "distinct_permnos": np.nan},
    {"stage": "CCM matched", "distinct_issues": np.nan, "distinct_issuers": np.nan, "distinct_gvkeys": len(sets.get("ccm_gvkeys", set())), "distinct_permnos": len(sets.get("ccm_permnos", set()))},
    {"stage": "CRSP monthly matched", "distinct_issues": np.nan, "distinct_issuers": np.nan, "distinct_gvkeys": np.nan, "distinct_permnos": len(sets.get("ccm_permnos", set()) & sets.get("crsp_monthly_permnos", set()))},
    {"stage": "CRSP daily matched", "distinct_issues": np.nan, "distinct_issuers": np.nan, "distinct_gvkeys": np.nan, "distinct_permnos": np.nan},
    {"stage": "IBES matched", "distinct_issues": np.nan, "distinct_issuers": np.nan, "distinct_gvkeys": np.nan, "distinct_permnos": np.nan},
]
attrition = pd.DataFrame(stages)
for col in ["distinct_issues", "distinct_issuers", "distinct_gvkeys", "distinct_permnos"]:
    vals = attrition[col]
    attrition[f"{col}_retained_from_prior"] = vals / vals.shift(1)
display_compact(attrition, n=20)
print("Where exact FISD issuer-to-Compustat linkage is unavailable, fields are left blank rather than inferred.")

,stage,distinct_issues,distinct_issuers,distinct_gvkeys,distinct_permnos,distinct_issues_retained_from_prior,distinct_issuers_retained_from_prior,distinct_gvkeys_retained_from_prior,distinct_permnos_retained_from_prior
0,FISD rated issues,298562.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FISD rated issuers,NaN,11907.0,NaN,NaN,NaN,NaN,NaN,NaN
2,issue-issuer mapping,298562.0,11908.0,NaN,NaN,NaN,1.000084,NaN,NaN
3,bond-CRSP matched,NaN,NaN,NaN,351717.0,NaN,NaN,NaN,NaN
4,Compustat matched,NaN,NaN,34289.0,NaN,NaN,NaN,NaN,NaN
5,CCM matched,NaN,NaN,36147.0,36938.0,NaN,NaN,1.054186,NaN
6,CRSP monthly matched,NaN,NaN,NaN,24627.0,NaN,NaN,NaN,0.666712
7,CRSP daily matched,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,IBES matched,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Where exact FISD issuer-to-Compustat linkage is unavailable, fields are left blank rather than inferred.


In [43]:
# Timing and leakage checklist
checks = [
    ("Compustat availability should use rdq", "REVIEW", "Raw audit reports rdq missingness and rdq < datadate; feature code must use rdq availability."),
    ("ratings known by prediction date", "REVIEW", "Use rating_date and avoid future rating actions."),
    ("CCM link validity dates", "REVIEW", "Raw CCM links are unfiltered; feature code must apply linkdt/linkenddt."),
    ("CRSP features cut off at quarter-end", "REVIEW", "Raw CRSP is daily/monthly; feature code must enforce quarter-end cutoff."),
    ("IBES uses statpers", "PASS", "statsum_epsus date preference selects statpers."),
    ("current amount_outstanding not used historically", "REVIEW", "Use only with as_of_date semantics or exclude from historical features."),
    ("delisting returns incorporated later", "REVIEW", "Raw delistings are archived; return construction later must include dlret."),
    ("macro releases aligned to vintage/availability", "REVIEW", "Raw macro dates are not release-vintage adjusted here."),
    ("no observations after 2024 in main sample", "REVIEW", "Raw post-2024 files may exist; modeling sample must cap at 2024."),
    ("strict 2000-2018 train / 2019-2024 test wall", "REVIEW", "Enforce in downstream feature/model code."),
]
checklist = pd.DataFrame(checks, columns=["item", "status", "note"])
display(checklist)

,item,status,note
0,Compustat availability should use rdq,REVIEW,Raw audit reports rdq missingness and rdq < da...
1,ratings known by prediction date,REVIEW,Use rating_date and avoid future rating actions.
2,CCM link validity dates,REVIEW,Raw CCM links are unfiltered; feature code mus...
3,CRSP features cut off at quarter-end,REVIEW,Raw CRSP is daily/monthly; feature code must e...
4,IBES uses statpers,PASS,statsum_epsus date preference selects statpers.
5,current amount_outstanding not used historically,REVIEW,Use only with as_of_date semantics or exclude ...
6,delisting returns incorporated later,REVIEW,Raw delistings are archived; return constructi...
7,macro releases aligned to vintage/availability,REVIEW,Raw macro dates are not release-vintage adjust...
8,no observations after 2024 in main sample,REVIEW,Raw post-2024 files may exist; modeling sample...
9,strict 2000-2018 train / 2019-2024 test wall,REVIEW,Enforce in downstream feature/model code.


In [44]:
# Overall audit summary
flag_df = pd.DataFrame(AUDIT_FLAGS)
datasets = ["FISD", "Compustat", "CRSP", "Bond-CRSP", "Bond returns", "IBES", "macro"]
summary_rows = []
for ds in datasets:
    related = flag_df[flag_df["dataset"].astype(str).str.contains(ds.split()[0], case=False, na=False)] if not flag_df.empty else pd.DataFrame()
    status = "PASS" if related.empty else ("FAIL" if (related.get("severity") == "FAIL").any() else "REVIEW")
    summary_rows.append({
        "dataset": ds,
        "extraction_complete": "REVIEW" if ds in ["Bond returns", "macro"] else "PASS",
        "schema_valid": "REVIEW" if status != "PASS" else "PASS",
        "key_integrity": "REVIEW",
        "date_coverage": "REVIEW",
        "duplicate_issues": "REVIEW",
        "mergeability_checked": "PASS",
        "leakage_risk": "REVIEW",
        "status": status,
        "notes": "; ".join(related["issue"].head(3).astype(str)) if not related.empty else "No blocking issue flagged by notebook logic.",
    })
overall = pd.DataFrame(summary_rows)
display(overall)
if not flag_df.empty:
    display_compact(flag_df, n=50, title="Detailed flags")

,dataset,extraction_complete,schema_valid,key_integrity,date_coverage,duplicate_issues,mergeability_checked,leakage_risk,status,notes
0,FISD,PASS,REVIEW,REVIEW,REVIEW,REVIEW,PASS,REVIEW,REVIEW,Check Fitch records around 2014-2016; Do not u...
1,Compustat,PASS,PASS,REVIEW,REVIEW,REVIEW,PASS,REVIEW,PASS,No blocking issue flagged by notebook logic.
2,CRSP,PASS,PASS,REVIEW,REVIEW,REVIEW,PASS,REVIEW,PASS,No blocking issue flagged by notebook logic.
3,Bond-CRSP,PASS,PASS,REVIEW,REVIEW,REVIEW,PASS,REVIEW,PASS,No blocking issue flagged by notebook logic.
4,Bond returns,REVIEW,PASS,REVIEW,REVIEW,REVIEW,PASS,REVIEW,PASS,No blocking issue flagged by notebook logic.
5,IBES,PASS,PASS,REVIEW,REVIEW,REVIEW,PASS,REVIEW,PASS,No blocking issue flagged by notebook logic.
6,macro,REVIEW,PASS,REVIEW,REVIEW,REVIEW,PASS,REVIEW,PASS,No blocking issue flagged by notebook logic.


Detailed flags


,dataset,severity,issue,detail
0,FISD ratings,REVIEW,Check Fitch records around 2014-2016,"[{'year': 2014, 'rating_type': 'FR', 'rows': 8..."
1,FISD,REVIEW,Do not use current amount_outstanding historic...,Note: current amount_outstanding is a current/...


## Recommended Next Actions

Run the notebook top-to-bottom after each raw extraction batch and use the displayed `Overall audit summary` plus `Detailed flags` table to decide what blocks feature engineering.

Expected concrete follow-ups include:

- investigate anomalous Fitch records if the FISD ratings coverage cell flags 2014-2016 spikes
- resolve any missing or corrupt issuer/issue master files before issuer-level aggregation
- define valid CCM filters before CRSP/Compustat linking
- build or verify issuer-to-Compustat mapping rather than inferring one in the audit
- construct quarterly availability dates using `rdq`, rating dates, IBES `statpers`, and quarter-end CRSP cutoffs
- choose the issue-rating aggregation rule before building firm-quarter labels
- begin feature engineering only after blocking `FAIL` items are resolved